In [1]:
import csv, pandas

In [2]:
# get cannonical HH genes based on R-HSA Hedgehog 'On' State term
with open('Reactome_Pathways_2024.txt', newline='') as f:
    reader = csv.reader(f, delimiter='\t')   # adjust delimiter
    for row in reader:
        if row[0] == "Hedgehog 'On' State":
            canonical = list(set(row[2:-1]))
            print(row, len(row))
            print(canonical, len(canonical))   

["Hedgehog 'On' State", '', 'CDON', 'CSNK1A1', 'SMURF2', 'PTCH2', 'SMURF1', 'PTCH1', 'IQCE', 'KIF7', 'RBX1', 'PSMC6', 'PSMC4', 'PSMC5', 'GAS1', 'PSMC2', 'GPR161', 'PSMC3', 'PSMC1', 'ULK3', 'SPOPL', 'GAS8', 'NUMB', 'PSMD11', 'PSMD13', 'PSMD12', 'PSMD14', 'IHH', 'CDC73', 'GRK2', 'PSMD7', 'PSMD8', 'SHH', 'BOC', 'PSMD6', 'PSMD3', 'PSMD1', 'PSMD2', 'SPOP', 'RPS27A', 'ADRM1', 'ITCH', 'EVC', 'PSMA6', 'PSMA7', 'PSMA4', 'PSMA5', 'SEM1', 'PSMA2', 'PSMA3', 'SMO', 'PSMA1', 'DZIP1', 'UBA52', 'DHH', 'CUL3', 'HHIP', 'ARRB2', 'EFCAB7', 'GLI2', 'GLI1', 'GLI3', 'PSMB7', 'PSMB5', 'PSMB6', 'PSMB3', 'UBC', 'ARRB1', 'PSMB4', 'UBB', 'KIF3A', 'PSMB1', 'PSMB2', 'SUFU', 'EVC2', ''] 76
['GAS1', 'CUL3', 'PSMA4', 'EFCAB7', 'PSMD7', 'PSMD6', 'PSMD14', 'SUFU', 'PSMC4', 'PSMA6', 'SPOPL', 'UBB', 'HHIP', 'PSMD8', 'PSMC3', 'IQCE', 'ADRM1', 'NUMB', 'SEM1', 'PSMB7', 'PSMB3', 'GRK2', 'ARRB2', 'PSMC5', 'DZIP1', 'UBC', 'DHH', 'SPOP', 'PSMD2', 'RPS27A', 'SMURF1', 'PSMD12', 'PSMA7', 'ULK3', 'EVC', 'GLI2', 'EVC2', 'IHH', 'PSMB2

In [8]:
# get the non cannonical hh based on Table 1 from 

def appender(noncanonical):
    elements = set(row[2:-1])
    print('detected category with', len(elements))
    noncanonical = noncanonical | elements
    print('current', len(noncanonical))
    print()
    return noncanonical

noncanonical = set()

categories = [
    "RAF MAP Kinase Cascade", 'MAPK1 MAPK3 Signaling', 'MAPK1 (ERK2) Activation', 'MAPK3 (ERK1) Activation'
]

with open('Reactome_Pathways_2024.txt', newline='') as f:
    reader = csv.reader(f, delimiter='\t')   # adjust delimiter
    for row in reader:
        
        # RAS-RAF-MEK-ERK: RAF MAP Kinase Cascade, 

        for category in categories:
            if category == row[0]:
                noncanonical = appender(noncanonical)
            
        # MAPK1 MAPK3 Signaling
            
noncanonical = list(noncanonical)  
print(len(noncanonical), noncanonical[:10])

detected category with 9
current 9

detected category with 271
current 271

detected category with 10
current 271

detected category with 265
current 271

271 ['FGF10', 'FGF7', 'FGA', 'IL3RA', 'PPP2R5E', 'IQGAP1', 'RAP1A', 'SPRED2', 'CSF2RB', 'IL6ST']


In [26]:
# convert the two lists into ensembl
t2g_file = '/Users/adrian/software/kallisto/t2g.txt'
rosetta = pandas.read_csv(t2g_file, sep='\t', header=None)

ensembl_cannonical = []; ensembl_noncanonical = []

def converter(target):
    ensemblID = None
    conversion = list(set(rosetta[rosetta[2] == target][1].values))
    if len(conversion) == 0:
        pass
    elif len(conversion) == 1:
        ensemblID = conversion[0]
    else:
        print('ERROR')
        print(conversion)
    return ensemblID

for target in canonical:
    ensemblID = converter(target)
    if ensemblID is not None:
        ensembl_cannonical.append(ensemblID)
   
for target in noncanonical:
    ensemblID = converter(target)
    if ensemblID is not None:
        ensembl_noncanonical.append(ensemblID)

ensembl_cannonical = list(set(ensembl_cannonical))
ensembl_noncanonical = list(set(ensembl_noncanonical))
print(len(ensembl_cannonical), len(ensembl_noncanonical))

/var/folders/zc/gkbgbc397lzgnslfwph9pg_00000gn/T/ipykernel_2922/3559493961.py:3: DtypeWarning: Columns (0: 4) have mixed types. Specify dtype option on import or set low_memory=False.
  rosetta = pandas.read_csv(t2g_file, sep='\t', header=None)


73 269


In [27]:
expression_file = '/Users/adrian/research/bmcbf/038_husavik/results/DESeq2_TPM_values.transcripts.tsv'
df = pandas.read_csv(expression_file, sep='\t', index_col=0)
df

FileNotFoundError: [Errno 2] No such file or directory: '/Users/adrian/research/bmcbf/038_husavik/results/DESeq2_TPM_values.transcripts.tsv'

In [ ]:
sub = df.loc[uniqueIDs]
sub

In [ ]:
sub.index = targets
sub

In [ ]:
df_zscore = sub.apply(scipy.stats.zscore, axis=1)
df_zscore

In [ ]:
# euclidean ward seems to work fine
# correlation average also seems good
# correlation complete is the best with top 20
# cosine average good
g = seaborn.clustermap(
    df_zscore,
    metric='correlation',     # distance metric
    method='complete',       # clustering method (e.g., single, complete, average, ward)
    cmap='bwr',            # color map (others: 'coolwarm', 'viridis', 'mako', 'rocket')
    standard_scale=None,    # or 0 (rows) or 1 (columns) if you didn't z-score earlier
    figsize=(9, 14),
    #col_colors=group_colors,
    #row_colors=row_colors,
    dendrogram_ratio=(0.1, 0.1),  # adjust size of dendrograms
    cbar_pos=(-0.1, 0.8, 0.05, 0.18),  # colorbar position
    xticklabels=True,
    cbar_kws={'label': 'Z-score'},
    vmin=-2.5,
    vmax=2.5,
    yticklabels=True
)